## Iowa Optimization — Hierarchical M Experiments BLOCKGROUP

This notebook runs multi-district MIP optimization experiments FOR BLOCKGROUP.

| Metric | Edge weight | M |
|---|---|---|
| `hop_M` | 1 + {0, M, M²} | number of nodes |
| `euclidean_M` | eucl(i,j) + {0, M, M²} | two-sweep diameter |

The penalty tier is determined by GEOID20 prefix matching:
- `GEOID20[:11]` match → same tract (no penalty)
- `GEOID20[:5]` match → same county, different tract (+M)
- Otherwise → different county (+M²)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT_DIR))

import os

from src.read import read_graph_from_json
from src.experiment import run_optimization_experiment

os.makedirs("../../results", exist_ok=True)

### Experiment parameters


In [4]:
GRAPH_LEVELS = ["blockgroup"]
DISTANCE_METRICS = ["hop", "euclidean", "hop_M", "euclidean_M"]

K = 4
DEVIATIONS = [50]
CONTIGUITY = ["tree", "dist", "dag", "cut",]
TIME_LIMIT = 3600

OBJECTIVES = ["euclidean_moi", "cut_edges"]

total_combinations = len(GRAPH_LEVELS) * len(DISTANCE_METRICS)
print(f"Graph levels     : {GRAPH_LEVELS}")
print(f"Distance metrics : {DISTANCE_METRICS}")
print(f"Combinations     : {total_combinations}")
print(f"Experiments each : {len(DEVIATIONS) * len(CONTIGUITY)} × objectives")

Graph levels     : ['blockgroup']
Distance metrics : ['hop', 'euclidean', 'hop_M', 'euclidean_M']
Combinations     : 4
Experiments each : 4 × objectives


### Run all combinations

In [5]:
all_results = {}

for graph_level in GRAPH_LEVELS:
    print(f"Loading graph: IA_{graph_level}.json")
    G = read_graph_from_json(f"../../data/IA_{graph_level}.json", state="IA")
    has_coords = "X" in G.nodes[next(iter(G.nodes))]
    coord_key = "coords" if has_coords else "no_coords"
    print(
        f"  {G.number_of_nodes()} nodes, {G.number_of_edges()} edges | coords: {has_coords}"
    )

    for dist_metric in DISTANCE_METRICS:
        label = f"IA_{graph_level}_{dist_metric}"
        results_file = f"../../results/{label}_optimization.csv"

        df = run_optimization_experiment(
            G_base=G,
            deviations=DEVIATIONS,
            contiguity_models=CONTIGUITY,
            objectives=OBJECTIVES,
            k=K,
            distance_metric=dist_metric,
            time_limit=TIME_LIMIT,
            results_file=results_file,
        )

        df["graph_level"] = graph_level
        df["distance_metric"] = dist_metric
        all_results[label] = df

print(" All combinations complete.")

Loading graph: IA_blockgroup.json
  2703 nodes, 7325 edges | coords: True
Selected roots: [2375, 1533, 557, 1692]
Distance metric: hop

Total experiments: 8
Results will be saved to: ../../results/IA_blockgroup_hop_optimization.csv

[1/8] deviation=50 | contiguity=tree (dist: hop) | objective_type=euclidean_moi
Using L, U, k = 797543 797642 4
Set parameter Username
Academic license - for non-commercial use only - expires 2026-11-24
Set parameter OutputFlag to value 1
Set parameter LogToConsole to value 1
Set parameter MIPGap to value 0
Set parameter PoolSearchMode to value 0
Set parameter PoolSolutions to value 1
Set parameter TimeLimit to value 3600
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 22.6.0 22H730)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  3600
MIPGap  0
PoolSolutions  1

Optimize a model with 13519 rows, 10812 columns and 54036 nonzeros (Min)
Model fingerprint